# Random Forest Accidentologie (3 experiences)

Objectif:
- entrainer **3 RandomForest** avec variation d'hyperparametres
- comparer les performances sur le jeu de test
- exporter modeles + predictions + metadonnees pour un futur logging MLflow

Note: ce notebook **ne lance pas MLflow**.

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import gc
import json

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier


def find_project_root(marker: str = "out") -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    return Path.cwd()


ROOT = find_project_root("out")
DATA_PATH = ROOT / "out" / "accidents_model_ready_kept_with_time_bucket.csv"
ARTIFACT_DIR = ROOT / "out" / "rf_experiments"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "grave"
SEED = 42
CV_SPLITS = 3
N_ITER = 6  # reduit pour limiter la RAM (16 Go)
RF_N_JOBS = 1  # 1 seul job RF pour ne pas doubler la RAM
SEARCH_N_JOBS = 1  # pas de parallelisme imbrique
MAX_TRAIN_SAMPLES = 60_000  # reduit de 90k a 60k pour securiser la RAM

# Memes 15 features que le notebook 11 (CatBoost) pour comparaison equitable
PRODUCT15_V2 = [
    "dep",
    "lum",
    "atm",
    "catr",
    "agg",
    "int",
    "circ",
    "col",
    "vma_bucket",
    "catv_family_4",
    "manv_mode",
    "driver_age_bucket",
    "choc_mode",
    "driver_trajet_family",
    "time_bucket",
]

print("ROOT:", ROOT)
print("DATA_PATH:", DATA_PATH)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("Config:", {
    "cv_splits": CV_SPLITS,
    "n_iter": N_ITER,
    "rf_n_jobs": RF_N_JOBS,
    "search_n_jobs": SEARCH_N_JOBS,
    "max_train_samples": MAX_TRAIN_SAMPLES,
})

ROOT: /home/maxime/simplonalternance/alternance-CICDprediction
DATA_PATH: /home/maxime/simplonalternance/alternance-CICDprediction/out/accidents_model_ready_kept_with_time_bucket.csv
ARTIFACT_DIR: /home/maxime/simplonalternance/alternance-CICDprediction/out/rf_experiments
Config: {'cv_splits': 3, 'n_iter': 6, 'rf_n_jobs': 1, 'search_n_jobs': 1, 'max_train_samples': 60000}


In [2]:
assert DATA_PATH.exists(), f"Fichier introuvable: {DATA_PATH}"

read_cols = PRODUCT15_V2 + [TARGET]
df = pd.read_csv(
    DATA_PATH,
    sep=";",
    usecols=read_cols,
    dtype={c: "category" for c in PRODUCT15_V2},
    low_memory=False,
)

assert TARGET in df.columns, f"Colonne cible absente: {TARGET}"
missing_feats = [c for c in PRODUCT15_V2 if c not in df.columns]
assert not missing_feats, f"Colonnes manquantes dans le CSV: {missing_feats}"

df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = df[TARGET].astype("int8")

X = df[PRODUCT15_V2].copy()
y = df[TARGET].copy()

print("Shape:", X.shape)
print("Target rate (grave=1):", round(float(y.mean()), 4))
print("Features utilisees:", list(X.columns))

Shape: (164526, 15)
Target rate (grave=1): 0.3608
Features utilisees: ['dep', 'lum', 'atm', 'catr', 'agg', 'int', 'circ', 'col', 'vma_bucket', 'catv_family_4', 'manv_mode', 'driver_age_bucket', 'choc_mode', 'driver_trajet_family', 'time_bucket']


## Preprocessing

- Split train/val/test stratifie (60/20/20)
- **val** sert a optimiser le seuil, **test** sert a evaluer les metriques finales
- Imputation categorielle + OrdinalEncoder (pour RandomForest)
- Les 15 features sont toutes categorielles

In [3]:
# Split 60/20/20 : train / val (seuil) / test (evaluation finale)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=SEED, stratify=y_temp,
)

# Toutes les features product15_v2 sont categorielles
cat_cols = PRODUCT15_V2[:]
num_cols = []

num_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]
)

transformers = [("cat", cat_pipe, cat_cols)]
if num_cols:
    transformers.insert(0, ("num", num_pipe, num_cols))

preprocess = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
print("Features categorielles:", len(cat_cols))
print("Features numeriques:", len(num_cols))

Train: (98715, 15) | Val: (32905, 15) | Test: (32906, 15)
Features categorielles: 15
Features numeriques: 0


## Hyperparametres et experiences

On lance 3 recherches d'hyperparametres:
1. `rf_auc_opt` optimise `roc_auc`
2. `rf_f1_opt` optimise `f1`
3. `rf_recall_opt` optimise `recall`

Chaque recherche entraine un RandomForest different.

In [4]:
rf_base = RandomForestClassifier(
    random_state=SEED,
    n_jobs=RF_N_JOBS,
)


def make_search(scoring: str, param_dist: dict) -> RandomizedSearchCV:
    pipe = Pipeline(
        steps=[
            ("prep", preprocess),
            ("model", rf_base),
        ]
    )
    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=SEED)
    return RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_dist,
        n_iter=N_ITER,
        scoring=scoring,
        n_jobs=SEARCH_N_JOBS,
        pre_dispatch=SEARCH_N_JOBS,
        cv=cv,
        random_state=SEED,
        verbose=1,
        refit=True,
    )


# ---- Grilles securisees pour 16 Go RAM ----
# JAMAIS de max_depth=None (arbres infinis = RAM infinie)
# max_leaf_nodes en filet de securite pour borner la taille des arbres
# n_estimators max 300 (au lieu de 560)

PARAM_DIST_AUC = {
    "model__n_estimators": [100, 150, 200, 300],
    "model__max_depth": [12, 18, 24],
    "model__min_samples_split": [5, 10, 20],
    "model__min_samples_leaf": [2, 4, 8],
    "model__max_features": ["sqrt", "log2"],
    "model__max_leaf_nodes": [512, 1024, 2048],
    "model__bootstrap": [True],
    "model__max_samples": [0.6, 0.8],
    "model__class_weight": [None, "balanced", "balanced_subsample"],
}

PARAM_DIST_F1 = {
    "model__n_estimators": [100, 150, 200, 300],
    "model__max_depth": [10, 16, 22],
    "model__min_samples_split": [5, 10, 15],
    "model__min_samples_leaf": [2, 4, 6],
    "model__max_features": ["sqrt", "log2"],
    "model__max_leaf_nodes": [512, 1024, 2048],
    "model__bootstrap": [True],
    "model__max_samples": [0.6, 0.8],
    "model__class_weight": ["balanced", "balanced_subsample", {0: 1, 1: 2}],
}

PARAM_DIST_RECALL = {
    "model__n_estimators": [100, 150, 200, 300],
    "model__max_depth": [8, 12, 18],
    "model__min_samples_split": [5, 10, 20],
    "model__min_samples_leaf": [2, 4, 8],
    "model__max_features": ["sqrt", "log2"],
    "model__max_leaf_nodes": [512, 1024, 2048],
    "model__bootstrap": [True],
    "model__max_samples": [0.6, 0.8],
    "model__class_weight": ["balanced", "balanced_subsample", {0: 1, 1: 2}, {0: 1, 1: 3}],
}

EXPERIMENTS = [
    ("rf_auc_opt", "roc_auc", PARAM_DIST_AUC),
    ("rf_f1_opt", "f1", PARAM_DIST_F1),
    ("rf_recall_opt", "recall", PARAM_DIST_RECALL),
]


def evaluate_binary(y_true: pd.Series, proba: np.ndarray, threshold: float = 0.5) -> dict:
    pred = (proba >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, proba)),
        "pr_auc": float(average_precision_score(y_true, proba)),
    }


def best_threshold_by_f1(y_true: pd.Series, proba: np.ndarray) -> tuple:
    thresholds = np.linspace(0.05, 0.95, 91)
    rows_thr = [evaluate_binary(y_true, proba, float(t)) for t in thresholds]
    df_thr = pd.DataFrame(rows_thr)
    idx = int(df_thr["f1"].idxmax())
    best = df_thr.loc[idx].to_dict()
    return float(best["threshold"]), {k: float(v) for k, v in best.items()}

In [5]:
trained = {}
rows = []

if MAX_TRAIN_SAMPLES is not None and len(X_train) > MAX_TRAIN_SAMPLES:
    X_search, _, y_search, _ = train_test_split(
        X_train,
        y_train,
        train_size=MAX_TRAIN_SAMPLES,
        stratify=y_train,
        random_state=SEED,
    )
else:
    X_search, y_search = X_train, y_train

print(f"Search dataset: {X_search.shape}")

for name, scoring, param_dist in EXPERIMENTS:
    print(f"\n=== {name} | scoring={scoring} ===")
    search = make_search(scoring=scoring, param_dist=param_dist)
    search.fit(X_search, y_search)

    best_model = search.best_estimator_
    best_params = dict(search.best_params_)
    cv_best_score = float(search.best_score_)

    # Seuil optimise sur val (pas sur test)
    proba_val = best_model.predict_proba(X_val)[:, 1]
    best_thr, _ = best_threshold_by_f1(y_val, proba_val)

    # Metriques finales sur test (jamais vu par le modele ni le seuil)
    proba_test = best_model.predict_proba(X_test)[:, 1]
    metrics_05 = evaluate_binary(y_test, proba_test, threshold=0.5)
    metrics_best = evaluate_binary(y_test, proba_test, threshold=best_thr)

    trained[name] = {
        "model": best_model,
        "proba_test": proba_test,
        "metrics_05": metrics_05,
        "best_threshold": best_thr,
        "metrics_best": metrics_best,
        "best_params": best_params,
        "cv_best_score": cv_best_score,
    }

    rows.append({
        "run_name": name,
        "optimized_for": scoring,
        "cv_best_score": cv_best_score,
        "f1_05": metrics_05["f1"],
        "roc_auc_05": metrics_05["roc_auc"],
        "best_threshold_f1": best_thr,
        "f1_best": metrics_best["f1"],
        "best_params": best_params,
    })

    del search
    gc.collect()

results_df = pd.DataFrame(rows).sort_values(["f1_best", "roc_auc_05"], ascending=False).reset_index(drop=True)
results_df[["run_name", "optimized_for", "cv_best_score", "f1_05", "roc_auc_05", "f1_best", "best_threshold_f1"]]

Search dataset: (60000, 15)

=== rf_auc_opt | scoring=roc_auc ===
Fitting 3 folds for each of 6 candidates, totalling 18 fits

=== rf_f1_opt | scoring=f1 ===
Fitting 3 folds for each of 6 candidates, totalling 18 fits

=== rf_recall_opt | scoring=recall ===
Fitting 3 folds for each of 6 candidates, totalling 18 fits


,run_name,optimized_for,cv_best_score,f1_05,roc_auc_05,f1_best,best_threshold_f1
0,rf_auc_opt,roc_auc,0.797159,0.607859,0.802834,0.668106,0.35
1,rf_recall_opt,recall,0.826796,0.656772,0.798900,0.665634,0.57
2,rf_f1_opt,f1,0.657151,0.663574,0.799489,0.665156,0.49


In [6]:
registry = []

for item in rows:
    name = item["run_name"]
    model = trained[name]["model"]
    proba_test = trained[name]["proba_test"]
    best_thr = trained[name]["best_threshold"]

    model_path = ARTIFACT_DIR / f"{name}.joblib"
    pred_path = ARTIFACT_DIR / f"{name}_predictions.csv"
    meta_path = ARTIFACT_DIR / f"{name}_meta.json"

    joblib.dump(model, model_path)

    pred_df = pd.DataFrame({
        "y_true": y_test.to_numpy(),
        "proba": proba_test,
        "pred_05": (proba_test >= 0.5).astype(int),
        "pred_best_f1": (proba_test >= best_thr).astype(int),
    })
    pred_df.to_csv(pred_path, index=False)

    meta = {
        "run_name": name,
        "optimized_for": item["optimized_for"],
        "dataset": str(DATA_PATH),
        "target": TARGET,
        "seed": SEED,
        "cv_splits": CV_SPLITS,
        "n_iter": N_ITER,
        "cv_best_score": item["cv_best_score"],
        "threshold_05": 0.5,
        "best_threshold_f1": best_thr,
        "metrics_05": trained[name]["metrics_05"],
        "metrics_best_f1": trained[name]["metrics_best"],
        "best_params": item["best_params"],
        "model_path": str(model_path),
        "predictions_path": str(pred_path),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    registry.append({
        "run_name": name,
        "model_path": str(model_path),
        "predictions_path": str(pred_path),
        "meta_path": str(meta_path),
    })

registry_df = pd.DataFrame(registry)
registry_df


,run_name,model_path,predictions_path,meta_path
0,rf_auc_opt,/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...
1,rf_f1_opt,/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...
2,rf_recall_opt,/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...


## Preparation MLflow (sans execution)

Le tableau ci-dessous rassemble ce qu'il faudra logger dans MLflow plus tard.

Aucune commande MLflow n'est executee ici.

In [7]:
mlflow_ready_df = results_df.merge(registry_df, on="run_name", how="left")
mlflow_ready_df

,run_name,optimized_for,cv_best_score,f1_05,roc_auc_05,best_threshold_f1,f1_best,best_params,model_path,predictions_path,meta_path
0,rf_auc_opt,roc_auc,0.797159,0.607859,0.802834,0.35,0.668106,"{'model__n_estimators': 200, 'model__min_sampl...",/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...
1,rf_recall_opt,recall,0.826796,0.656772,0.798900,0.57,0.665634,"{'model__n_estimators': 200, 'model__min_sampl...",/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...
2,rf_f1_opt,f1,0.657151,0.663574,0.799489,0.49,0.665156,"{'model__n_estimators': 300, 'model__min_sampl...",/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...


In [8]:
best_idx = results_df["f1_best"].idxmax()
best_run = results_df.loc[best_idx, "run_name"]
best_params = results_df.loc[best_idx, "best_params"]

print("Best run (par F1):", best_run)
print("Best params:")
print(best_params)

Best run (par F1): rf_auc_opt
Best params:
{'model__n_estimators': 200, 'model__min_samples_split': 20, 'model__min_samples_leaf': 8, 'model__max_samples': 0.8, 'model__max_leaf_nodes': 2048, 'model__max_features': 'log2', 'model__max_depth': 24, 'model__class_weight': None, 'model__bootstrap': True}


## Integration MLflow (activee a la demande)

Ce bloc log les 3 RandomForest dans MLflow quand `ENABLE_MLFLOW=True`.

In [ ]:
ENABLE_MLFLOW = True
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "accidentologie_model_benchmark"
ENABLE_MODEL_REGISTRY = True
REGISTERED_MODEL_NAME = "briefml-rf-product15-v2-time-bucket"

if ENABLE_MLFLOW:
    import numpy as np
    import mlflow
    import mlflow.sklearn
    from mlflow.tracking import MlflowClient

    def _to_params(d):
        out = {}
        if isinstance(d, dict):
            for k, v in d.items():
                if isinstance(v, (int, float, str, bool, np.integer, np.floating, np.bool_)):
                    out[k] = v.item() if hasattr(v, "item") else v
        return out

    def _normalize_metrics(d, prefix="valid_"):
        keys = {"accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "threshold"}
        out = {}
        if isinstance(d, dict):
            for k, v in d.items():
                if k in keys and isinstance(v, (int, float, np.integer, np.floating)):
                    out[f"{prefix}{k}"] = float(v)
        return out

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

    # On enregistre le meilleur run (par f1_best) dans le registry
    best_run_name = results_df.loc[results_df["f1_best"].idxmax(), "run_name"]

    for row in rows:
        run_name = row["run_name"]
        payload = trained.get(run_name, {})
        model = payload.get("model")

        if model is None:
            print(f"[mlflow] skip {run_name}: model manquant")
            continue

        metrics_05 = _normalize_metrics(payload.get("metrics_05"), prefix="valid_")
        metrics_best = _normalize_metrics(payload.get("metrics_best"), prefix="valid_bestf1_")

        model_path = ARTIFACT_DIR / f"{run_name}.joblib"
        pred_path = ARTIFACT_DIR / f"{run_name}_predictions.csv"
        meta_path = ARTIFACT_DIR / f"{run_name}_meta.json"

        # Enregistrer dans le registry uniquement le meilleur run
        register_name = REGISTERED_MODEL_NAME if (ENABLE_MODEL_REGISTRY and run_name == best_run_name) else None

        with mlflow.start_run(run_name=run_name):
            mlflow.set_tags({
                "notebook": "12_random_forest_mlflow_prep.ipynb",
                "model_family": "random_forest",
                "model_flavor": "sklearn",
                "tag": "accidentologie",
                "optimized_for": str(row.get("optimized_for", "unknown")),
            })
            if register_name:
                mlflow.set_tag("registry_enabled", "true")

            mlflow.log_param("seed", int(SEED))
            mlflow.log_param("cv_splits", int(CV_SPLITS))
            mlflow.log_param("n_iter", int(N_ITER))
            mlflow.log_param("target", TARGET)
            mlflow.log_param("cv_primary_metric", str(row.get("optimized_for", "unknown")))
            mlflow.log_params(_to_params(row.get("best_params", {})))
            cv_primary_score = row.get("cv_best_score")
            if cv_primary_score is not None:
                mlflow.log_metric("cv_primary_score", float(cv_primary_score))

            if metrics_05:
                mlflow.log_metrics(metrics_05)
            if metrics_best:
                mlflow.log_metrics(metrics_best)

            model_info = mlflow.sklearn.log_model(
                model,
                artifact_path="model",
                registered_model_name=register_name,
            )

            for p, art in [(model_path, "models"), (pred_path, "predictions"), (meta_path, "metadata")]:
                if p.exists():
                    mlflow.log_artifact(str(p), artifact_path=art)

            # --- Tags de performance sur le Model Registry ---
            if register_name and getattr(model_info, "registered_model_version", None):
                version = model_info.registered_model_version
                perf_tags = {
                    "f1_best": f"{payload['metrics_best']['f1']:.4f}",
                    "roc_auc": f"{payload['metrics_05']['roc_auc']:.4f}",
                    "recall_best": f"{payload['metrics_best']['recall']:.4f}",
                    "precision_best": f"{payload['metrics_best']['precision']:.4f}",
                    "best_threshold": f"{payload['best_threshold']:.2f}",
                    "optimized_for": str(row.get("optimized_for", "unknown")),
                }
                for tag_key, tag_val in perf_tags.items():
                    client.set_model_version_tag(register_name, version, tag_key, tag_val)
                print(f"[mlflow] model registered: {register_name} v{version}")
                print(f"[mlflow] registry tags: {perf_tags}")

            print(f"[mlflow] logged: {run_name}")
else:
    print("MLflow desactive. Passe ENABLE_MLFLOW=True quand le serveur sera pret.")